# HD cross-match: Gaia homogeneous results vs Ralf target list

Compare `results/Gaia_homogeneous_target_selection_2026.06.24.xlsx` with `data/2ES_targetlist_astrid_export_2024Dec_comments.xlsx` on HD number.

Ralf rows are filtered to `prio != 3` and `magV < 8` (same magnitude cut as the Gaia query).

Uses `_normalize_star_id_for_merge()` so `HD170493` and `HD 170493` match.

In [3]:
import re

import pandas as pd

from config import DATA_DIRECTORY, RESULTS_DIRECTORY
from core.data_processing import _normalize_star_id_for_merge

STR_COLUMNS = ['source_id', 'source_id_dr2', 'source_id_dr3', 'HD Number', 'GJ Number', 'HIP Number']
GAIA_FILE = f'{RESULTS_DIRECTORY}Gaia_homogeneous_target_selection_2026.06.24.xlsx'
GAIA_DATE = '2026.06.24'

RALF_FILE = f'{DATA_DIRECTORY}2ES_targetlist_astrid_export_2024Dec_comments.xlsx'
STAR_ID_COL = 'star_ID  '
MAGV_COL = 'magV     '
MAGV_MAX = 8

gaia = pd.read_excel(GAIA_FILE, dtype={col: str for col in STR_COLUMNS})
ralf = pd.read_excel(RALF_FILE, engine='openpyxl', header=1)
ralf = ralf[ralf['prio'] != 3]  # same as merge_and_format_stellar_data()
ralf[MAGV_COL] = pd.to_numeric(ralf[MAGV_COL], errors='coerce')
ralf = ralf[ralf[MAGV_COL] < MAGV_MAX]

gaia_hd = gaia[
    gaia['HD Number'].notna() & (gaia['HD Number'].astype(str).str.strip() != '')
].copy()
ralf_hd = ralf[ralf[STAR_ID_COL].astype(str).str.match(r'(?i)^HD', na=False)].copy()

gaia_hd['_hd_norm'] = gaia_hd['HD Number'].apply(_normalize_star_id_for_merge)
ralf_hd['_hd_norm'] = ralf_hd[STAR_ID_COL].apply(_normalize_star_id_for_merge)

gaia_ids = set(gaia_hd['_hd_norm'])
ralf_ids = set(ralf_hd['_hd_norm'])

only_gaia_ids = sorted(gaia_ids - ralf_ids)
only_ralf_ids = sorted(ralf_ids - gaia_ids)

only_gaia = gaia_hd[gaia_hd['_hd_norm'].isin(only_gaia_ids)].sort_values('_hd_norm')
only_ralf = ralf_hd[ralf_hd['_hd_norm'].isin(only_ralf_ids)].sort_values('_hd_norm')

print(f'HD stars in Gaia homogeneous ({GAIA_DATE}): {len(gaia_ids)}')
print(f'HD stars in Ralf (magV < {MAGV_MAX}):        {len(ralf_ids)}')
print(f'In both:                                     {len(gaia_ids & ralf_ids)}')
print(f'Only in Gaia homogeneous:                    {len(only_gaia_ids)}')
print(f'Only in Ralf target list:                    {len(only_ralf_ids)}')

only_gaia_path = f'{RESULTS_DIRECTORY}hd_only_in_Gaia_homogeneous_target_selection_{GAIA_DATE}.xlsx'
only_ralf_path = f'{RESULTS_DIRECTORY}hd_only_in_Ralf_targetlist_magV{MAGV_MAX}.xlsx'
only_gaia.to_excel(only_gaia_path, index=False)
only_ralf.to_excel(only_ralf_path, index=False)
print(f'\nSaved {only_gaia_path}')
print(f'Saved {only_ralf_path}')

preview_cols = ['HD Number', '_hd_norm', 'source_id', 'T_eff [K]', 'HZ Detection Limit [M_Earth]']
only_gaia[preview_cols].head(10)

Maintenance with possible short-time disconnections: 29 June 2026 18:00–20:00 CEST
HD stars in Gaia homogeneous (2026.06.24): 1031
HD stars in Ralf (magV < 8):        141
In both:                                     65
Only in Gaia homogeneous:                    966
Only in Ralf target list:                    76

Saved ../results/hd_only_in_Gaia_homogeneous_target_selection_2026.06.24.xlsx
Saved ../results/hd_only_in_Ralf_targetlist_magV8.xlsx


,HD Number,_hd_norm,source_id,T_eff [K],HZ Detection Limit [M_Earth]
151,HD 10002,HD 10002,Gaia DR3 5023104341621773184,5174.744141,0.955642
389,HD 100180,HD 100180,Gaia DR3 3966121308211393920,5917.371094,1.301230
600,HD 1002,HD 1002,Gaia DR3 2323040096922657920,5754.399414,1.568938
791,HD 100286,HD 100286,Gaia DR3 3482326708703712640,6047.772949,2.091774
746,HD 100287,HD 100287,Gaia DR3 3482326708703712896,6019.297363,1.861510
988,HD 100339,HD 100339,Gaia DR3 3974220242141965056,6224.566406,10.488538
992,HD 1004,HD 1004,Gaia DR3 4922906809055232640,6373.297852,11.047291
226,HD 100555,HD 100555,Gaia DR3 5370497853621034368,5499.673340,1.089578
5,HD 100623,HD 100623,Gaia DR3 3478127463341507072,5225.102539,0.469718
656,HD 101,HD 101,Gaia DR3 2797111130991722240,5889.315430,1.648997


## HZ detection limit vs temperature (Gaia homogeneous vs Ralf overlap)

Cross-match Ralf's list (`prio != 3`, `magV < 8`) to `Gaia_homogeneous_target_selection_2026.06.24.xlsx` using the same ID normalization as `merge_and_format_stellar_data()` (HD / HIP / GJ).

Plot style matches `HZ_detection_limit_vs_temperature_zoomed_4.png`:
- **Yellow circles:** full Gaia homogeneous sample (`T_eff [K]` vs `HZ Detection Limit [M_Earth]`)
- **Red +:** Ralf stars that match Gaia, with Teff and detection limit taken from the Gaia file

In [14]:
import matplotlib.pyplot as plt
import numpy as np

from config import FIGURES_DIRECTORY

MERGE_KEYS = ['HD Number', 'HIP Number', 'GJ Number']

gaia_xmatch = pd.read_excel(GAIA_FILE, dtype={col: str for col in STR_COLUMNS})
ralf_xmatch = pd.read_excel(RALF_FILE, engine='openpyxl', header=1)
ralf_xmatch = ralf_xmatch[ralf_xmatch['prio'] != 3]
ralf_xmatch[MAGV_COL] = pd.to_numeric(ralf_xmatch[MAGV_COL], errors='coerce')
ralf_xmatch = ralf_xmatch[ralf_xmatch[MAGV_COL] < MAGV_MAX].copy()

ralf_xmatch['_id_norm'] = ralf_xmatch[STAR_ID_COL].apply(_normalize_star_id_for_merge)

gaia_xmatch['HIP Number'] = gaia_xmatch['HIP Number'].apply(
    lambda x: f'HIP{x}' if pd.notna(x) and x != '' and not str(x).startswith('HIP') else x
)
for key in MERGE_KEYS:
    gaia_xmatch[f'{key}_norm'] = gaia_xmatch[key].apply(_normalize_star_id_for_merge)

overlap_parts = [
    ralf_xmatch.merge(
        gaia_xmatch,
        left_on='_id_norm',
        right_on=f'{key}_norm',
        how='inner',
    )
    for key in MERGE_KEYS
]
gaia_ralf_overlap = pd.concat(overlap_parts, ignore_index=True)
gaia_ralf_overlap = gaia_ralf_overlap.drop_duplicates(subset=STAR_ID_COL, keep='first')

plot_gaia = gaia_xmatch.dropna(subset=['T_eff [K]', 'HZ Detection Limit [M_Earth]']).copy()
plot_overlap = gaia_ralf_overlap.dropna(subset=['T_eff [K]', 'HZ Detection Limit [M_Earth]']).copy()

print(f'Gaia homogeneous sample: {len(gaia_xmatch)} stars')
print(f'Ralf targets (magV < {MAGV_MAX}): {len(ralf_xmatch)} stars')
print(f'Cross-matched (Ralf ∩ Gaia): {len(gaia_ralf_overlap)} stars')
print(f'Plotted overlap points: {len(plot_overlap)} stars')

color = plt.cm.viridis(np.linspace(0, 1, 8))[7]
xmin = min(plot_gaia['T_eff [K]'].min(), plot_overlap['T_eff [K]'].min()) - 100
output_path = f'{FIGURES_DIRECTORY}HZ_detection_limit_vs_temperature_zoomed_4_Gaia_homogeneous_{GAIA_DATE}_ralf_overlap.png'

# Increase circle and cross size
CIRCLE_SIZE = 60
CROSS_SIZE = 50

# Manual matplotlib to allow custom legend
fig, ax = plt.subplots(figsize=(8, 6), dpi=150)
# Plot Gaia sample (all stars)
scatter_gaia = ax.scatter(
    plot_gaia['T_eff [K]'], plot_gaia['HZ Detection Limit [M_Earth]'],
    c=[color], alpha=0.7, s=CIRCLE_SIZE, label="Filtered Gaia sample"
)
# Plot Ralf sample (overlap)
scatter_overlap = ax.scatter(
    plot_overlap['T_eff [K]'], plot_overlap['HZ Detection Limit [M_Earth]'],
    color='red', marker='x', s=CROSS_SIZE, label="Curated HARPS sample"
)

# Draw vertical dashed line at Sun's temperature (5772 K)
SUN_TEFF = 5772
ax.axvline(SUN_TEFF, color='black', linestyle='--', linewidth=2, alpha=0.5)
# Annotate "Sun" horizontally and to the left of the dashed line
ax.annotate(
    "Sun",
    xy=(SUN_TEFF, 0.2),
    xycoords='data',
    xytext=(-5, 0),  # shift left of the line
    textcoords='offset points',
    ha='right',
    va='center',
    fontsize=14,
    color='black',
    alpha=0.5,
    rotation=0,
    fontweight='bold'
)
       

# --- Increase font size throughout the plot ---
LARGE_FONT = 18
MEDIUM_FONT = 16
SMALL_FONT = 14

# ax labels
ax.set_xlabel('Stellar Temperature (K)', fontsize=LARGE_FONT)
ax.set_ylabel(r'HZ Detection Limit ($M_\oplus$)', fontsize=LARGE_FONT)
ax.set_xlim((xmin, 6300))
ax.set_ylim((0, 4))
ax.grid(True, linestyle='--', alpha=0.7)

# xtick and ytick fonts
ax.tick_params(axis='both', which='major', labelsize=MEDIUM_FONT)

# Set x and y ticks every 500 (T_eff every 500K, y every 0.5)
from matplotlib.ticker import MultipleLocator
ax.xaxis.set_major_locator(MultipleLocator(500))
ax.yaxis.set_major_locator(MultipleLocator(0.5))

# Legend (make sure only unique labels, include Sun)
handles, labels = ax.get_legend_handles_labels()
from collections import OrderedDict
by_label = OrderedDict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), fontsize=MEDIUM_FONT)

plt.tight_layout()
plt.savefig(output_path)
plt.show()

gaia_ralf_overlap_path = f'{RESULTS_DIRECTORY}gaia_ralf_crossmatch_{GAIA_DATE}.xlsx'
gaia_ralf_overlap.to_excel(gaia_ralf_overlap_path, index=False)
print(f'Saved {gaia_ralf_overlap_path}')

Gaia homogeneous sample: 1037 stars
Ralf targets (magV < 8): 142 stars
Cross-matched (Ralf ∩ Gaia): 65 stars
Plotted overlap points: 65 stars
Saved ../results/gaia_ralf_crossmatch_2026.06.24.xlsx
